In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [3]:
a = pd.read_csv('./123.csv')
a

,Unnamed: 0,Jul-19,Aug-19,Sep-19,Oct-19,Nov-19,Dec-19,Jan-20,Feb-20,Mar-20,...,Oct-Dec'20,Jan-Mar'21,Apr-Jun'21,Jul-Sept'21,Oct-Dec'21,Jan-Mar'22,Apr-Jun'22,Jul-Sept'22,Oct-Dec'22,Jan-Mar'23
0,Android,"262,500","262,500","437,500","437,500","525,000","525,000","612,500","612,500","612,500",...,"4,375,000","4,375,000","8,750,000","6,125,000","5,250,000","6,125,000","13,125,000","7,000,000","6,125,000","6,125,000"
1,iOS,"37,500","37,500","62,500","62,500","75,000","75,000","87,500","87,500","87,500",...,"625,000","625,000","1,250,000","875,000","750,000","875,000","1,875,000","1,000,000","875,000","875,000"
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Months,Year,D30,D60,D90,D120,D150,D180,D210,D240,...,D450,D480,D510,D540,D570,D600,D630,D660,D690,D720
4,Android Retention Rate,FY19,10.0%,8.0%,5.0%,4.0%,3.0%,2.5%,2.5%,2.5%,...,1.0%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%
5,NaN,FY20,11.0%,9.0%,6.0%,5.0%,4.0%,3.5%,3.5%,3.5%,...,2.0%,1.5%,1.5%,1.5%,1.5%,1.5%,1.5%,1.5%,1.5%,1.5%
6,NaN,FY21,12.5%,10.5%,7.5%,6.5%,5.5%,5.0%,5.0%,5.0%,...,3.5%,3.0%,3.0%,3.0%,3.0%,3.0%,3.0%,3.0%,3.0%,3.0%
7,NaN,FY22,14.0%,12.0%,9.0%,8.0%,7.0%,6.5%,6.5%,6.5%,...,5.0%,4.5%,4.5%,4.5%,4.5%,4.5%,4.5%,4.5%,4.5%,4.5%
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,iOS Retention Rate,FY19,12.0%,10.0%,8.0%,6.0%,5.0%,4.0%,3.0%,2.0%,...,1.0%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%,0.5%


In [7]:
bc = a.iloc[4:8,(2)].tolist()
bc

['10.0%', '11.0%', '12.5%', '14.0%']

In [4]:
b = a.transpose()
new_header = b.iloc[0]
b = b[1:]
b.columns = new_header

In [ ]:
dlist = []
for i in range(24):
    d30 = (a.iloc[4:8,(i+2)].tolist())
    d30 = [item for item in d30 for j in range(12)]
    d30 = d30[3:(3+45)]
    dlist.append(d30)
dlist

In [8]:
c = b[['Android']]
c['Android'] = c['Android'].str.replace(',','')
c['Android'] = c['Android'].astype(int)

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  
/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [9]:
e = c.loc["Oct-Dec'20":]

split = e['Android'].tolist()
split[:] = [x // 3 for x in split]
split = [item for item in split for i in range(3)]
x11 = pd.DataFrame({'Android':split})

In [10]:
from datetime import timedelta, date
x1 = c.loc[:"Sep-20"]
y = pd.concat([x1,x11],ignore_index=True)

In [13]:
def daterange(date1, date2):
    for n in range(int ((date2 - date1).days)+1):
        yield date1 + timedelta(n)
date_list = []
start_dt = date(19,7,1)
end_dt = date(23,3,30)
for dt in daterange(start_dt, end_dt):
    date_list.append(dt.strftime("%Y-%m"))
    
xy = pd.DataFrame({'months': date_list})
xy.drop_duplicates('months',inplace = True)
xy.reset_index(drop=True, inplace=True)

In [14]:
z = pd.merge(y, xy , left_index=True,right_index = True)
z.set_index('months',inplace=True)

In [16]:
zero = []
for i in range(24):
    temp = []
    for j in range(i+1):
        temp.append(0)
    zero.append(temp)
anusers = z['Android'].tolist()
columns = []
for i in range(24):
    temp1 = zero[i] + anusers
    temp1 = temp1[:45]
    columns.append(temp1)

In [17]:
dlist1 = []
for i in range(24):
    t1 = zero[i] + dlist[i]
    t1 = t1[:45]
    dlist1.append(t1)

In [12]:
import itertools, pandas
c1 = pd.DataFrame((_ for _ in itertools.zip_longest(*columns)), columns=['c1', 'c2', 'c3','c4','c5','c6','c7','c8','c9','c10','c11','c12','c13','c14','c15','c16','c17','c18','c19','c20','c21','c22','c23','c24'])
#c1.loc[:,'Total'] = c1.sum(axis=1)

In [13]:
import itertools, pandas
c2 = pd.DataFrame((_ for _ in itertools.zip_longest(*dlist1)), columns=['c1', 'c2', 'c3','c4','c5','c6','c7','c8','c9','c10','c11','c12','c13','c14','c15','c16','c17','c18','c19','c20','c21','c22','c23','c24'])

c2 = c2.replace('%','',regex=True).astype('float')/100

In [14]:
c3 = pd.DataFrame(c1.values*c2.values, columns=c1.columns, index=c1.index)

c4 = c3.sum(axis=1)
c4 = c4.to_frame().reset_index()
c4.columns = ['index','totals']
c4 = c4[['totals']]
c4 = c4.set_index(z.index)

In [15]:
d = pd.merge(z, c4, left_index=True, right_index=True)
d = d.apply(pd.to_numeric)

In [16]:
e = d.sum(axis=1)
e = e.to_frame().reset_index()
e.columns = ['month','total number of active android users']
e = e.astype(int, errors='ignore')
e.set_index('month',inplace = True)
f = pd.merge(d, e, left_index=True,right_index = True)
f.columns = ['new users','retained users','total android users']
f

,new users,retained users,total android users
months,,,
19-07,262500,0.000,262500
19-08,262500,26250.000,288750
19-09,437500,47250.000,484750
19-10,437500,77875.000,515375
19-11,525000,102375.000,627375
19-12,525000,127750.000,652750
20-01,612500,148312.500,760812
20-02,612500,173250.000,785750
20-03,612500,194687.500,807187
